In [1]:
!pip -q install -U google-genai pymupdf faiss-cpu pillow numpy pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 22.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [2]:
import os
import io
import re
import json
import time
import hashlib
import traceback

import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm

import fitz  # PyMuPDF
import faiss

from google import genai
from google.genai import types

from google.colab import files

In [3]:
import os
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("API key loaded successfully.")

Enter your Gemini API Key: ··········
API key loaded successfully.


In [4]:
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [5]:
# ==============================
# DOCURAG CONFIGURATION
# ==============================

EMBED_MODEL = "gemini-embedding-2"
GENERATION_MODEL = "gemini-3.8-flash"

# Recommended by Google's embedding documentation
EMBED_DIM = 768

# Folder containing uploaded documents
DATA_DIR = "/content/docurag_data"

# Folder where PDF pages will be rendered
PAGE_DIR = "/content/docurag_pages"

# Persistent index files
INDEX_PATH = "/content/docurag.index"
METADATA_PATH = "/content/docurag_metadata.json"

# Retrieval configuration
INITIAL_TOP_K = 15
FINAL_TOP_K = 6

# Minimum confidence required
CONFIDENCE_THRESHOLD = 40.0

# Create directories
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PAGE_DIR, exist_ok=True)

print("Configuration loaded.")

Configuration loaded.


In [6]:
!pip -q install -U kagglehub

In [7]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [10]:
import kagglehub
import os

PDFVQA_DIR = "/content/pdfvqa"

os.makedirs(
    PDFVQA_DIR,
    exist_ok=True
)

pdfvqa_path = kagglehub.competition_download(
    "pdfvqa",
    output_dir=PDFVQA_DIR
)

print("PDF-VQA downloaded to:")
print(pdfvqa_path)

100%|██████████| 2.05G/2.05G [00:35<00:00, 62.8MB/s]

Extracting files...


PDF-VQA downloaded to:
/content/pdfvqa


In [11]:
for root, dirs, files_list in os.walk(
    PDFVQA_DIR
):

    level = root.replace(
        PDFVQA_DIR,
        ""
    ).count(os.sep)

    indent = "  " * level

    print(
        indent +
        os.path.basename(root) +
        "/"
    )

    for filename in files_list[:10]:

        print(
            indent +
            "  " +
            filename
        )

pdfvqa/
  train_doc_info.pkl
  train_dataframe.csv
  val_dataframe.csv
  test_dataframe_without_answer.csv
  test_doc_info.pkl
  val_doc_info.pkl
  .complete/
    competitions/
      pdfvqa/
        bundle.complete
  test_images/
    test_images/
      24223258.pdf_0.png
      29445036.pdf_5.png
      29664013.pdf_8.png
      27909739.pdf_2.png
      29899075.pdf_4.png
      20825657.pdf_4.png
      27582126.pdf_6.png
      21745410.pdf_6.png
      29651637.pdf_5.png
      26576319.pdf_3.png
  train_images/
    train_images/
      23671879.pdf_2.png
      28915816.pdf_11.png
      28391368.pdf_8.png
      28729654.pdf_6.png
      29616292.pdf_8.png
      26286317.pdf_1.png
      28527130.pdf_24.png
      27527613.pdf_1.png
      23226972.pdf_5.png
      22293957.pdf_1.png
  val_images/
    val_images/
      19775471.pdf_1.png
      25812680.pdf_2.png
      26577720.pdf_3.png
      24381752.pdf_6.png
      28975906.pdf_8.png
      25983105.pdf_5.png
      26348214.pdf_19.png
      27367

In [12]:
# Find the important PDF-VQA files and folders

import os

def find_file(filename):
    for root, dirs, files in os.walk(PDFVQA_DIR):
        if filename in files:
            return os.path.join(root, filename)
    return None

def find_directory(dirname):
    for root, dirs, files in os.walk(PDFVQA_DIR):
        if dirname in dirs:
            return os.path.join(root, dirname)
    return None


TRAIN_CSV = find_file("train_dataframe.csv")
VAL_CSV = find_file("val_dataframe.csv")
TRAIN_PKL = find_file("train_doc_info.pkl")
VAL_PKL = find_file("val_doc_info.pkl")

TRAIN_IMAGES = find_directory("train_images")
VAL_IMAGES = find_directory("val_images")

print("TRAIN CSV   :", TRAIN_CSV)
print("VAL CSV     :", VAL_CSV)
print("TRAIN PKL   :", TRAIN_PKL)
print("VAL PKL     :", VAL_PKL)
print("TRAIN IMAGES:", TRAIN_IMAGES)
print("VAL IMAGES  :", VAL_IMAGES)

TRAIN CSV   : /content/pdfvqa/train_dataframe.csv
VAL CSV     : /content/pdfvqa/val_dataframe.csv
TRAIN PKL   : /content/pdfvqa/train_doc_info.pkl
VAL PKL     : /content/pdfvqa/val_doc_info.pkl
TRAIN IMAGES: /content/pdfvqa/train_images
VAL IMAGES  : /content/pdfvqa/val_images


In [13]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nSample questions:")
display(train_df.head(10))

Train shape: (3951, 6)
Validation shape: (581, 6)

Columns:
['Unnamed: 0', 'pmcid', 'question', 'answer', 'global_id', 'question_type']

Sample questions:


,Unnamed: 0,pmcid,question,answer,global_id,question_type
0,0,27923395,Name out the section that include the Fig. 2.,['Expression of Ascl1 and neuronal markers in ...,"[243838, 243824]",parent_relationship_understanding
1,1,27923395,"When you search for the description of Fig. 3,...",['Differential expression of proneural genes w...,"[243867, 243824, 243849, 243866, 243878]",parent_relationship_understanding
2,2,27923395,Which section does include the description of ...,"['Discussion', 'Results', 'Expression of neuro...","[243877, 243824, 243825, 243884, 243849]",parent_relationship_understanding
3,3,27923395,Where can you find the Table 3?,['Differential expression of proneural genes w...,"[243867, 243849, 243824]",parent_relationship_understanding
4,4,27923395,"Where is the 'Oka C et al,1995' cited in the d...",['Expression of Ascl1 and neuronal markers in ...,"[243838, 243824]",parent_relationship_understanding
5,5,27923395,"When you check the Background, which subsectio...",['No subsection!'],[-1],child_relationship_understanding
6,6,21772848,What is the 2. The Unrestricted Rise of Lung C...,['No subsection!'],[-1],child_relationship_understanding
7,7,28630713,Which section does describe the Table 1?,['Results'],[235459],parent_relationship_understanding
8,8,28630713,When you search for the description of Table 2...,['Results'],[235459],parent_relationship_understanding
9,9,28630713,"Where can 'Gervais KJ et al,2012' be found in ...",['Background'],[235451],parent_relationship_understanding


In [14]:
print("Unique training documents:", train_df["pmcid"].nunique())
print("Unique validation documents:", val_df["pmcid"].nunique())

print("\nExample document IDs:")
print(train_df["pmcid"].dropna().unique()[:20])

Unique training documents: 800
Unique validation documents: 115

Example document IDs:
[27923395 21772848 28630713 28634532 25291521 26438282 22938389 25093011
 22546084 19838333 22046523 20637102 24772164 26862038 29386049 27896644
 27965885 27429732 28985410 29860382]


In [15]:
MAX_DOCUMENTS = 50

selected_documents = (
    train_df["pmcid"]
    .dropna()
    .drop_duplicates()
    .head(MAX_DOCUMENTS)
    .tolist()
)

working_df = train_df[
    train_df["pmcid"].isin(selected_documents)
].copy()

print("Selected documents:", len(selected_documents))
print("Questions in working dataset:", len(working_df))

display(working_df.head(10))

Selected documents: 50
Questions in working dataset: 238


,Unnamed: 0,pmcid,question,answer,global_id,question_type
0,0,27923395,Name out the section that include the Fig. 2.,['Expression of Ascl1 and neuronal markers in ...,"[243838, 243824]",parent_relationship_understanding
1,1,27923395,"When you search for the description of Fig. 3,...",['Differential expression of proneural genes w...,"[243867, 243824, 243849, 243866, 243878]",parent_relationship_understanding
2,2,27923395,Which section does include the description of ...,"['Discussion', 'Results', 'Expression of neuro...","[243877, 243824, 243825, 243884, 243849]",parent_relationship_understanding
3,3,27923395,Where can you find the Table 3?,['Differential expression of proneural genes w...,"[243867, 243849, 243824]",parent_relationship_understanding
4,4,27923395,"Where is the 'Oka C et al,1995' cited in the d...",['Expression of Ascl1 and neuronal markers in ...,"[243838, 243824]",parent_relationship_understanding
5,5,27923395,"When you check the Background, which subsectio...",['No subsection!'],[-1],child_relationship_understanding
6,6,21772848,What is the 2. The Unrestricted Rise of Lung C...,['No subsection!'],[-1],child_relationship_understanding
7,7,28630713,Which section does describe the Table 1?,['Results'],[235459],parent_relationship_understanding
8,8,28630713,When you search for the description of Table 2...,['Results'],[235459],parent_relationship_understanding
9,9,28630713,"Where can 'Gervais KJ et al,2012' be found in ...",['Background'],[235451],parent_relationship_understanding


In [16]:
sample_images = []

for root, dirs, files in os.walk(TRAIN_IMAGES):
    for filename in files:
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            sample_images.append(os.path.join(root, filename))

            if len(sample_images) >= 20:
                break

    if len(sample_images) >= 20:
        break

print("Found images:", len(sample_images))

for img in sample_images:
    print(img)

Found images: 20
/content/pdfvqa/train_images/train_images/23671879.pdf_2.png
/content/pdfvqa/train_images/train_images/28915816.pdf_11.png
/content/pdfvqa/train_images/train_images/28391368.pdf_8.png
/content/pdfvqa/train_images/train_images/28729654.pdf_6.png
/content/pdfvqa/train_images/train_images/29616292.pdf_8.png
/content/pdfvqa/train_images/train_images/26286317.pdf_1.png
/content/pdfvqa/train_images/train_images/28527130.pdf_24.png
/content/pdfvqa/train_images/train_images/27527613.pdf_1.png
/content/pdfvqa/train_images/train_images/23226972.pdf_5.png
/content/pdfvqa/train_images/train_images/22293957.pdf_1.png
/content/pdfvqa/train_images/train_images/24392284.pdf_62.png
/content/pdfvqa/train_images/train_images/26139832.pdf_1.png
/content/pdfvqa/train_images/train_images/29052018.pdf_2.png
/content/pdfvqa/train_images/train_images/27600869.pdf_10.png
/content/pdfvqa/train_images/train_images/25789552.pdf_4.png
/content/pdfvqa/train_images/train_images/29062544.pdf_5.png
/co

In [17]:
print("TRAIN PKL   :", TRAIN_PKL)
print("TRAIN IMAGES:", TRAIN_IMAGES)

TRAIN PKL   : /content/pdfvqa/train_doc_info.pkl
TRAIN IMAGES: /content/pdfvqa/train_images


In [18]:
class EvidenceNode:

    def __init__(
        self,
        node_id,
        document,
        page,
        content_type,
        text="",
        image_path=None
    ):

        self.node_id = node_id
        self.document = document
        self.page = page
        self.content_type = content_type
        self.text = text
        self.image_path = image_path

    def to_dict(self):

        return {
            "node_id": self.node_id,
            "document": self.document,
            "page": self.page,
            "content_type": self.content_type,
            "text": self.text,
            "image_path": self.image_path
        }

    def __repr__(self):

        return (
            f"EvidenceNode("
            f"{self.document}, "
            f"page={self.page}, "
            f"type={self.content_type})"
        )

In [19]:
def clean_text(text):

    if not text:
        return ""

    text = text.replace("\x00", " ")

    # Normalize whitespace
    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    # Normalize excessive newlines
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()

In [20]:
def chunk_text(
    text,
    chunk_size=1200,
    overlap=200
):

    text = clean_text(text)

    if not text:
        return []

    chunks = []

    start = 0

    while start < len(text):

        end = min(
            start + chunk_size,
            len(text)
        )

        chunk = text[start:end]

        # Try to end at a natural boundary
        if end < len(text):

            candidates = [
                chunk.rfind("\n\n"),
                chunk.rfind(". "),
                chunk.rfind(" "),
            ]

            best_break = max(candidates)

            if best_break > chunk_size * 0.55:

                end = start + best_break

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(
            end - overlap,
            start + 1
        )

    return chunks

In [21]:
def render_pdf_page(
    page,
    output_path,
    dpi=150
):

    matrix = fitz.Matrix(
        dpi / 72,
        dpi / 72
    )

    pix = page.get_pixmap(
        matrix=matrix,
        alpha=False
    )

    pix.save(output_path)

    return output_path

In [22]:
def process_pdf(pdf_path):

    document_name = os.path.basename(
        pdf_path
    )

    pdf = fitz.open(pdf_path)

    nodes = []

    document_hash = hashlib.md5(
        document_name.encode()
    ).hexdigest()[:10]

    document_folder = os.path.join(
        PAGE_DIR,
        document_hash
    )

    os.makedirs(
        document_folder,
        exist_ok=True
    )

    print(
        f"\nProcessing {document_name} "
        f"({len(pdf)} pages)"
    )

    for page_index in tqdm(
        range(len(pdf)),
        desc=document_name
    ):

        page = pdf[page_index]

        page_number = page_index + 1

        # ==========================
        # TEXT EXTRACTION
        # ==========================

        raw_text = page.get_text("text")

        chunks = chunk_text(raw_text)

        for chunk_index, chunk in enumerate(chunks):

            node_id = (
                f"{document_name}_"
                f"p{page_number}_"
                f"text_{chunk_index}"
            )

            nodes.append(
                EvidenceNode(
                    node_id=node_id,
                    document=document_name,
                    page=page_number,
                    content_type="text",
                    text=chunk
                )
            )

        # ==========================
        # PAGE IMAGE
        # ==========================

        image_path = os.path.join(
            document_folder,
            f"page_{page_number}.png"
        )

        render_pdf_page(
            page,
            image_path
        )

        visual_node_id = (
            f"{document_name}_"
            f"p{page_number}_visual"
        )

        nodes.append(
            EvidenceNode(
                node_id=visual_node_id,
                document=document_name,
                page=page_number,
                content_type="image",
                image_path=image_path
            )
        )

    pdf.close()

    return nodes

In [23]:
def process_image(image_path):

    filename = os.path.basename(
        image_path
    )

    node = EvidenceNode(
        node_id=f"{filename}_visual",
        document=filename,
        page=1,
        content_type="image",
        image_path=image_path
    )

    return [node]

In [24]:
all_nodes = []

for filename in sorted(
    os.listdir(DATA_DIR)
):

    path = os.path.join(
        DATA_DIR,
        filename
    )

    if not os.path.isfile(path):
        continue

    extension = (
        filename.lower()
        .split(".")[-1]
    )

    try:

        if extension == "pdf":

            nodes = process_pdf(path)

        elif extension in [
            "png",
            "jpg",
            "jpeg"
        ]:

            print(
                f"\nProcessing image: "
                f"{filename}"
            )

            nodes = process_image(path)

        else:

            print(
                f"Skipping unsupported file: "
                f"{filename}"
            )

            continue

        all_nodes.extend(nodes)

    except Exception as e:

        print(
            f"Error processing {filename}: {e}"
        )

        traceback.print_exc()

print("\n==============================")
print("INGESTION COMPLETE")
print("==============================")
print("Total evidence nodes:", len(all_nodes))


INGESTION COMPLETE
Total evidence nodes: 0


In [25]:
for node in all_nodes[:20]:

    print(
        node.document,
        "| Page:", node.page,
        "| Type:", node.content_type
    )

    if node.text:
        print(
            node.text[:250]
        )

    print("-" * 70)

In [26]:
def embed_text(text):

    result = client.models.embed_content(

        model=EMBED_MODEL,

        contents=(
            "title: document evidence | "
            f"text: {text}"
        ),

        config=types.EmbedContentConfig(
            output_dimensionality=EMBED_DIM
        )
    )

    vector = np.array(
        result.embeddings[0].values,
        dtype=np.float32
    )

    return vector

In [27]:
def get_mime_type(image_path):

    extension = (
        image_path
        .lower()
        .split(".")[-1]
    )

    if extension == "jpg":
        return "image/jpeg"

    if extension == "jpeg":
        return "image/jpeg"

    return "image/png"


def embed_image(image_path):

    with open(
        image_path,
        "rb"
    ) as f:

        image_bytes = f.read()

    mime_type = get_mime_type(
        image_path
    )

    result = client.models.embed_content(

        model=EMBED_MODEL,

        contents=[
            types.Part.from_bytes(
                data=image_bytes,
                mime_type=mime_type
            )
        ],

        config=types.EmbedContentConfig(
            output_dimensionality=EMBED_DIM
        )
    )

    vector = np.array(
        result.embeddings[0].values,
        dtype=np.float32
    )

    return vector

In [28]:
def get_mime_type(image_path):

    extension = (
        image_path
        .lower()
        .split(".")[-1]
    )

    if extension == "jpg":
        return "image/jpeg"

    if extension == "jpeg":
        return "image/jpeg"

    return "image/png"


def embed_image(image_path):

    with open(
        image_path,
        "rb"
    ) as f:

        image_bytes = f.read()

    mime_type = get_mime_type(
        image_path
    )

    result = client.models.embed_content(

        model=EMBED_MODEL,

        contents=[
            types.Part.from_bytes(
                data=image_bytes,
                mime_type=mime_type
            )
        ],

        config=types.EmbedContentConfig(
            output_dimensionality=EMBED_DIM
        )
    )

    vector = np.array(
        result.embeddings[0].values,
        dtype=np.float32
    )

    return vector

In [29]:
def normalize_vector(vector):

    vector = np.asarray(
        vector,
        dtype=np.float32
    )

    norm = np.linalg.norm(vector)

    if norm == 0:
        return vector

    return vector / norm

In [30]:
vectors = []
metadata = []

failed_nodes = []

print(
    f"Creating embeddings for "
    f"{len(all_nodes)} nodes..."
)

for node in tqdm(
    all_nodes,
    desc="Embedding"
):

    try:

        if node.content_type == "text":

            vector = embed_text(
                node.text
            )

        elif node.content_type == "image":

            vector = embed_image(
                node.image_path
            )

        else:

            continue

        vector = normalize_vector(
            vector
        )

        vectors.append(vector)

        metadata.append(
            node.to_dict()
        )

        # Small delay to reduce rate-limit risk
        time.sleep(0.05)

    except Exception as e:

        failed_nodes.append(
            {
                "node_id": node.node_id,
                "error": str(e)
            }
        )

        print(
            f"\nEmbedding failed: "
            f"{node.node_id}"
        )

print("\nEmbedding complete.")
print("Successful:", len(vectors))
print("Failed:", len(failed_nodes))

Creating embeddings for 0 nodes...


Embedding: 0it [00:00, ?it/s]


Embedding complete.
Successful: 0
Failed: 0


In [32]:
if len(vectors) == 0:

    raise RuntimeError(
        "No embeddings were created."
    )

vectors = np.vstack(
    vectors
).astype(np.float32)

print(
    "Vector matrix shape:",
    vectors.shape
)

RuntimeError: No embeddings were created.

In [ ]:
index = faiss.IndexFlatIP(
    EMBED_DIM
)

index.add(
    vectors
)

print(
    "FAISS index created."
)

print(
    "Number of vectors:",
    index.ntotal
)

In [ ]:
faiss.write_index(
    index,
    INDEX_PATH
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Index saved.")
print("Metadata saved.")

In [ ]:
if os.path.exists(INDEX_PATH):

    index = faiss.read_index(
        INDEX_PATH
    )

    with open(
        METADATA_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        metadata = json.load(f)

    print(
        "Existing DocuRAG index loaded."
    )

    print(
        "Vectors:",
        index.ntotal
    )

In [ ]:
def embed_query(query):

    result = client.models.embed_content(

        model=EMBED_MODEL,

        contents=(
            "task: search result | "
            f"query: {query}"
        ),

        config=types.EmbedContentConfig(
            output_dimensionality=EMBED_DIM
        )
    )

    vector = np.array(
        result.embeddings[0].values,
        dtype=np.float32
    )

    return normalize_vector(
        vector
    )

In [ ]:
def retrieve(
    query,
    top_k=INITIAL_TOP_K
):

    query_vector = embed_query(
        query
    )

    scores, indices = index.search(
        query_vector.reshape(1, -1),
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx < 0:
            continue

        item = metadata[idx].copy()

        item["similarity"] = float(
            score
        )

        item["rank"] = len(results) + 1

        results.append(item)

    return results

In [ ]:
test_query = (
    "What is the main architecture "
    "described in the document?"
)

results = retrieve(
    test_query,
    top_k=10
)

for result in results:

    print(
        f"Rank {result['rank']} | "
        f"Score {result['similarity']:.4f} | "
        f"{result['document']} | "
        f"Page {result['page']} | "
        f"{result['content_type']}"
    )

    if result["text"]:

        print(
            result["text"][:350]
        )

    print("-" * 90)

In [ ]:
def classify_query(query):

    prompt = f"""
Classify the following document question.

Possible categories:

TEXT
VISUAL
TABLE
MIXED

Definitions:

TEXT:
Question primarily requiring textual evidence.

VISUAL:
Question primarily requiring diagrams, figures,
images, charts, or visual relationships.

TABLE:
Question primarily requiring numerical/table evidence.

MIXED:
Question requires both textual and visual evidence.

Question:
{query}

Return ONLY one category.
"""

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt
    )

    category = (
        response.text
        .strip()
        .upper()
    )

    allowed = [
        "TEXT",
        "VISUAL",
        "TABLE",
        "MIXED"
    ]

    if category not in allowed:
        category = "MIXED"

    return category

In [ ]:
def rerank_results(
    query,
    results,
    final_k=FINAL_TOP_K
):

    if len(results) <= final_k:
        return results

    candidate_blocks = []

    for i, result in enumerate(results):

        if result["content_type"] == "text":

            content = result["text"]

        else:

            content = (
                "Visual evidence from "
                f"{result['document']} "
                f"page {result['page']}."
            )

        candidate_blocks.append(
            f"""
CANDIDATE {i}

Document: {result['document']}
Page: {result['page']}
Type: {result['content_type']}
Vector similarity: {result['similarity']:.4f}

Evidence:
{content}
"""
        )

    prompt = f"""
You are an evidence reranking system.

User question:
{query}

Rank the candidates by their usefulness for
answering the user's question.

Consider:

1. Semantic relevance.
2. Directness of evidence.
3. Specificity.
4. Whether the evidence actually supports
   an answer rather than merely mentioning
   the topic.

Candidates:

{"".join(candidate_blocks)}

Return ONLY valid JSON.

Format:

[
  {{"candidate": 0, "score": 95}},
  {{"candidate": 3, "score": 90}}
]

Return all candidates.
"""

    try:

        response = client.models.generate_content(
            model=GENERATION_MODEL,
            contents=prompt
        )

        text = response.text.strip()

        # Remove markdown fences if present
        text = re.sub(
            r"```json|```",
            "",
            text
        ).strip()

        ranking = json.loads(
            text
        )

        ranking = sorted(
            ranking,
            key=lambda x: x["score"],
            reverse=True
        )

        reranked = []

        for item in ranking:

            candidate_index = int(
                item["candidate"]
            )

            if (
                0 <= candidate_index
                < len(results)
            ):

                result = results[
                    candidate_index
                ].copy()

                result["rerank_score"] = (
                    float(item["score"])
                )

                reranked.append(
                    result
                )

        return reranked[:final_k]

    except Exception as e:

        print(
            "Reranking failed:",
            e
        )

        # Safe fallback
        return results[:final_k]

In [ ]:
def build_evidence_context(
    results
):

    text_parts = []

    image_results = []

    for i, result in enumerate(results):

        source_id = i + 1

        citation = (
            f"[SOURCE {source_id}] "
            f"{result['document']} "
            f"- Page {result['page']}"
        )

        if result["content_type"] == "text":

            text_parts.append(
                f"""
{citation}

Type: Text

Evidence:
{result['text']}
"""
            )

        elif result["content_type"] == "image":

            image_results.append(
                {
                    **result,
                    "source_id": source_id
                }
            )

            text_parts.append(
                f"""
{citation}

Type: Visual evidence

The original page image is supplied
to the model separately.
"""
            )

    context = "\n".join(
        text_parts
    )

    return (
        context,
        image_results
    )

In [ ]:
SYSTEM_PROMPT = """
You are DocuRAG, a multimodal enterprise
document intelligence system.

Your job is to answer questions ONLY using
the retrieved evidence provided to you.

STRICT RULES:

1. Never invent facts.

2. Every factual claim must be supported
   by retrieved evidence.

3. Cite evidence using:
   [SOURCE X]

4. Never fabricate a source or page number.

5. If visual evidence is relevant, use the
   supplied page image.

6. If the evidence is insufficient, explicitly
   say that the evidence is insufficient.

7. Do not use your general world knowledge
   to fill gaps.

8. Clearly distinguish between:
   - directly stated information
   - interpretation
   - uncertainty

9. If multiple sources agree, mention that
   the evidence is corroborated.

10. If sources conflict, explicitly identify
    the conflict.

11. Answer concisely but with enough technical
    detail to be useful.

12. Do not cite sources that do not support
    the claim.

OUTPUT FORMAT:

ANSWER:
<grounded answer>

EVIDENCE ANALYSIS:
<why the retrieved evidence supports the answer>

SOURCES:
- [SOURCE X] <document> - Page <number>
- [SOURCE Y] <document> - Page <number>

CONFIDENCE:
<number from 0 to 100>

ABSTAIN:
YES or NO
"""

In [ ]:
def calculate_confidence(
    results
):

    if not results:
        return 0.0

    similarities = np.array(
        [
            r["similarity"]
            for r in results
        ],
        dtype=np.float32
    )

    # Clip because similarity can vary
    # depending on embedding space.
    similarities = np.clip(
        similarities,
        0,
        1
    )

    top_score = float(
        similarities[0]
    )

    mean_score = float(
        np.mean(similarities)
    )

    # Evidence consistency
    consistency = float(
        np.mean(
            similarities
            >= max(
                0.01,
                top_score * 0.80
            )
        )
    )

    confidence = (
        0.50 * top_score +
        0.30 * mean_score +
        0.20 * consistency
    )

    confidence *= 100

    return round(
        float(
            np.clip(
                confidence,
                0,
                100
            )
        ),
        1
    )

In [ ]:
def should_abstain(
    results,
    threshold=CONFIDENCE_THRESHOLD
):

    if not results:
        return True

    confidence = calculate_confidence(
        results
    )

    return confidence < threshold

In [ ]:
def generate_grounded_answer(
    query,
    results
):

    evidence_context, image_results = (
        build_evidence_context(
            results
        )
    )

    prompt = f"""
{SYSTEM_PROMPT}

USER QUESTION:
{query}

RETRIEVED EVIDENCE:

{evidence_context}
"""

    contents = [
        prompt
    ]

    # Attach relevant page images
    for item in image_results[:4]:

        try:

            with open(
                item["image_path"],
                "rb"
            ) as f:

                image_bytes = f.read()

            mime_type = get_mime_type(
                item["image_path"]
            )

            contents.append(
                types.Part.from_bytes(
                    data=image_bytes,
                    mime_type=mime_type
                )
            )

        except Exception as e:

            print(
                "Could not attach image:",
                e
            )

    response = client.models.generate_content(

        model=GENERATION_MODEL,

        contents=contents,

        config=types.GenerateContentConfig(
            temperature=0.1
        )
    )

    return response.text

In [ ]:
def docurag(
    query,
    initial_top_k=INITIAL_TOP_K,
    final_top_k=FINAL_TOP_K
):

    start_time = time.time()

    # ==========================
    # 1. QUERY UNDERSTANDING
    # ==========================

    query_type = classify_query(
        query
    )

    # ==========================
    # 2. VECTOR RETRIEVAL
    # ==========================

    candidates = retrieve(
        query,
        top_k=initial_top_k
    )

    if not candidates:

        return {
            "answer": (
                "I could not find any "
                "relevant evidence."
            ),
            "confidence": 0,
            "query_type": query_type,
            "sources": [],
            "latency": (
                time.time() - start_time
            )
        }

    # ==========================
    # 3. RERANKING
    # ==========================

    final_results = rerank_results(
        query,
        candidates,
        final_k=final_top_k
    )

    # ==========================
    # 4. CONFIDENCE
    # ==========================

    confidence = calculate_confidence(
        final_results
    )

    # ==========================
    # 5. ABSTENTION
    # ==========================

    if confidence < CONFIDENCE_THRESHOLD:

        return {
            "answer": (
                "I could not find enough "
                "reliable evidence in the "
                "indexed documents to answer "
                "this question confidently."
            ),
            "confidence": confidence,
            "query_type": query_type,
            "sources": final_results,
            "latency": (
                time.time() - start_time
            ),
            "abstained": True
        }

    # ==========================
    # 6. GROUNDED GENERATION
    # ==========================

    answer = generate_grounded_answer(
        query,
        final_results
    )

    latency = (
        time.time() - start_time
    )

    return {
        "answer": answer,
        "confidence": confidence,
        "query_type": query_type,
        "sources": final_results,
        "latency": latency,
        "abstained": False
    }

In [ ]:
question = """
What is the main architecture described
in the documents?
"""

result = docurag(
    question
)

print("=" * 80)
print("DOCURAG ANSWER")
print("=" * 80)

print(result["answer"])

print("\n")
print("=" * 80)
print("SYSTEM METRICS")
print("=" * 80)

print(
    "Query Type:",
    result["query_type"]
)

print(
    "Confidence:",
    result["confidence"],
    "%"
)

print(
    "Latency:",
    round(
        result["latency"],
        2
    ),
    "seconds"
)

print(
    "Abstained:",
    result["abstained"]
)

In [ ]:
print(
    "\n" + "=" * 80
)

print(
    "RETRIEVED SOURCES"
)

print(
    "=" * 80
)

for source in result["sources"]:

    print(
        f"\nRank: "
        f"{source.get('rank', '-')}"
    )

    print(
        "Document:",
        source["document"]
    )

    print(
        "Page:",
        source["page"]
    )

    print(
        "Type:",
        source["content_type"]
    )

    print(
        "Vector similarity:",
        round(
            source["similarity"],
            4
        )
    )

    if "rerank_score" in source:

        print(
            "Rerank score:",
            source["rerank_score"]
        )

    if source["text"]:

        print(
            "Text:",
            source["text"][:500]
        )

    print("-" * 80)

In [ ]:
import matplotlib.pyplot as plt

def show_source_image(
    source
):

    if (
        source["content_type"]
        != "image"
    ):

        print(
            "This source does not "
            "contain an image."
        )

        return

    image = Image.open(
        source["image_path"]
    )

    plt.figure(
        figsize=(12, 8)
    )

    plt.imshow(image)

    plt.axis("off")

    plt.title(
        f"{source['document']} "
        f"- Page {source['page']}"
    )

    plt.show()

In [ ]:
# Show the first retrieved visual source

for source in result["sources"]:

    if source["content_type"] == "image":

        show_source_image(
            source
        )

        break

In [ ]:
questions = [

    "What is the main objective of this document?",

    "What architecture is proposed?",

    "What are the major components of the system?",

    "Explain the retrieval process.",

    "What does the main architecture diagram show?",

    "What dataset was used?",

    "What are the key results?",

]

for question in questions:

    print("\n")
    print("=" * 100)
    print("QUESTION:")
    print(question)
    print("=" * 100)

    result = docurag(
        question
    )

    print(
        result["answer"]
    )

    print(
        "\nConfidence:",
        result["confidence"],
        "%"
    )

    print(
        "Latency:",
        round(
            result["latency"],
            2
        ),
        "seconds"
    )

In [ ]:
fake_question = """
Who is the current President of Mars
and what is their favorite programming language?
"""

result = docurag(
    fake_question
)

print(result["answer"])

print(
    "\nConfidence:",
    result["confidence"],
    "%"
)

print(
    "Abstained:",
    result["abstained"]
)

In [ ]:
evaluation_questions = [

    {
        "question":
        "What is the main objective of the document?",

        "expected_keywords":
        [
            "objective",
            "system"
        ]
    },

    {
        "question":
        "What architecture is proposed?",

        "expected_keywords":
        [
            "architecture"
        ]
    },

    {
        "question":
        "What are the main components?",

        "expected_keywords":
        [
            "component"
        ]
    },

]

In [ ]:
evaluation_results = []

for item in evaluation_questions:

    start = time.time()

    result = docurag(
        item["question"]
    )

    answer_lower = (
        result["answer"]
        .lower()
    )

    matched = sum(
        keyword.lower()
        in answer_lower
        for keyword
        in item["expected_keywords"]
    )

    keyword_score = (
        matched /
        max(
            1,
            len(
                item["expected_keywords"]
            )
        )
    )

    evaluation_results.append({

        "question":
            item["question"],

        "confidence":
            result["confidence"],

        "latency":
            result["latency"],

        "keyword_score":
            keyword_score,

        "abstained":
            result["abstained"]
    })


evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df

In [ ]:
print(
    "Average confidence:",
    round(
        evaluation_df[
            "confidence"
        ].mean(),
        2
    )
)

print(
    "Average latency:",
    round(
        evaluation_df[
            "latency"
        ].mean(),
        2
    ),
    "seconds"
)

print(
    "Average keyword score:",
    round(
        evaluation_df[
            "keyword_score"
        ].mean() * 100,
        2
    ),
    "%"
)

print(
    "Abstention rate:",
    round(
        evaluation_df[
            "abstained"
        ].mean() * 100,
        2
    ),
    "%"
)

In [ ]:
def ask_docurag(question):

    result = docurag(
        question
    )

    response = {

        "question":
            question,

        "answer":
            result["answer"],

        "confidence":
            result["confidence"],

        "query_type":
            result["query_type"],

        "latency_seconds":
            round(
                result["latency"],
                3
            ),

        "abstained":
            result["abstained"],

        "sources": []
    }

    for source in result["sources"]:

        response["sources"].append({

            "document":
                source["document"],

            "page":
                source["page"],

            "type":
                source["content_type"],

            "similarity":
                round(
                    source["similarity"],
                    4
                ),

            "rerank_score":
                source.get(
                    "rerank_score",
                    None
                )
        })

    return response

In [ ]:
response = ask_docurag(
    "Explain the proposed system architecture."
)

print(
    json.dumps(
        response,
        indent=2
    )
)

In [ ]:
result = ask_docurag(
    "What are the key findings?"
)

with open(
    "/content/docurag_result.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Result saved to:"
    "/content/docurag_result.json"
)